In [ ]:
from pathlib import Path
from typing import Literal

In [ ]:
import chatlas
import faicons
from playsound3 import playsound
from pyhere import here
from shiny import App, reactive, ui

In [ ]:
# Tools ------------------------------------------------------------------------
SoundChoice = Literal["correct", "incorrect", "new-round", "you-win"]

In [ ]:
sound_map: dict[SoundChoice, Path] = {
    "correct": here("data/sounds/smb_coin.wav"),
    "incorrect": here("data/sounds/wilhelm.wav"),
    "new-round": here("data/sounds/victory_fanfare_mono.wav"),
    "you-win": here("data/sounds/smb_stage_clear.wav"),
}

In [ ]:
def play_sound(sound: SoundChoice = "correct") -> str:
    """
    Plays a sound effect.

    Parameters
    ----------
    sound: Which sound effect to play: "correct", "incorrect", "new-round" or
           "you-win". Play the "new-round" sound after the user picks a theme
           for the round. Play the "correct" and "incorrect" sounds when the
           user answers a question correctly or incorrectly, respectively. And
           play the "you-win" sound at the end of a round of questions.

    Returns
    -------
    A confirmation that the sound was played.
    """
    if sound not in sound_map.keys():
        raise ValueError(
            f"sound must be one of {sorted(sound_map.keys())}; got {sound!r}"
        )

    playsound(sound_map[sound])

    return f"The '{sound}' sound was played."

UI ---------------------------------------------------------------------------

In [ ]:
app_ui = ui.page_fillable(
    ui.chat_ui("chat"),
)

In [ ]:
def server(input, output, session):
    chat_ui = ui.Chat(id="chat")

    # Set up the chat instance
    client = chatlas.ChatPosit(
        model="zai-org/GLM-5.3-Flash",
        system_prompt=here("_solutions/14_quiz-game-1/prompt.md").read_text(),
    )
    client.register_tool(
        play_sound,
        annotations={
            "title": "Play Sound Effect",
            "extra": {
                "icon": faicons.icon_svg("volume-high"),
            },
        },
    )

    @chat_ui.on_user_submit
    async def handle_user_input(user_input: str):
        # Use `content="all"` to include tool calls in the response stream
        response = await client.stream_async(user_input, content="all")
        await chat_ui.append_message_stream(response)

    @reactive.effect
    def _():
        # Start the game when the app launches
        chat_ui.update_user_input(value="Let's play the quiz game!", submit=True)

In [ ]:
app = App(app_ui, server)